# Behavioral Dynamics Pipeline Demo

**Full end-to-end example** using the modules we've built:

- `unified_pipeline.py`
- `topological_distance.py`
- `mapper_analyzer.py`
- `separatrix_detection.py`

This notebook demonstrates:
1. Generating synthetic CogEval-style traces
2. Running the full unified pipeline
3. Computing topological distance between regimes
4. Analyzing temporal persistence (dissipate / recur / propagate / metastasize)
5. Exporting data for the interactive visualizers

In [ ]:
import sys
import numpy as np
import json
from pathlib import Path

# Add repo root to path so `llm_eval` package is importable
sys.path.insert(0, str(Path.cwd().parent.parent))

from llm_eval.unified_pipeline import BehavioralDynamicsPipeline
from llm_eval.topological_distance import topological_distance
import matplotlib.pyplot as plt

## 1. Generate Synthetic Traces

We'll create traces that simulate a model going through different behavioral regimes, including a metastasis event.

In [ ]:
np.random.seed(42)
traces = []
debt = 0.12

for t in range(100):
    if 20 < t < 45:
        debt += 0.018
    elif 60 < t < 75:
        debt -= 0.025
    elif t > 82:
        debt += 0.012  # Metastasis

    traces.append({
        "turn": t,
        "refusal_score": min(1.0, 0.2 + debt * 0.8 + np.random.normal(0, 0.04)),
        "moralizing": min(1.0, 0.15 + debt * 0.75 + np.random.normal(0, 0.05)),
        "truth_score": max(0.0, 0.85 - debt * 0.9 + np.random.normal(0, 0.04)),
        "instruction_loyalty": max(0.0, 0.9 - debt * 0.85 + np.random.normal(0, 0.03)),
        "personality": max(0.0, 0.82 - debt * 0.7 + np.random.normal(0, 0.05)),
        "debt": round(max(0.0, min(1.0, debt)), 3)
    })

print(f"Generated {len(traces)} traces")

## 2. Run the Unified Pipeline

In [ ]:
pipeline = BehavioralDynamicsPipeline(traces)
report = pipeline.run_full_analysis()

print("=== Pipeline Summary ===")
print(json.dumps(report["summary"], indent=2))

## 3. Topological Distance Analysis

In [ ]:
# Create two clouds for comparison
early_traces = traces[:40]      # Mostly contract-aligned
late_traces = traces[70:]       # Mixed + metastasis

features_early = np.array([[t['refusal_score'], t['moralizing'], t['truth_score']] for t in early_traces])
features_late = np.array([[t['refusal_score'], t['moralizing'], t['truth_score']] for t in late_traces])

dist = topological_distance(features_early, features_late)
print(f"Topological distance between early and late windows: {dist:.4f}")

## 4. Visualize Debt Trajectory + Key Events

In [ ]:
turns = [t['turn'] for t in traces]
debt_values = [t['debt'] for t in traces]

plt.figure(figsize=(12, 5))
plt.plot(turns, debt_values, label='Debt', color='#00ff9f', linewidth=2)

# Mark approximate regime changes
plt.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='Regime shift begins')
plt.axvline(x=45, color='orange', linestyle='--', alpha=0.7)
plt.axvline(x=82, color='red', linestyle='--', alpha=0.8, label='Metastasis begins')

plt.title("Behavioral Debt Trajectory with Regime Changes")
plt.xlabel("Turn")
plt.ylabel("Debt")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Export for Interactive Visualizers

This generates the JSON files used by `temporal_persistence_visualizer.html` and `mapper_visualizer.html`.

In [ ]:
pipeline.export_for_visualizers("output")
print("Exported visualization data to 'output/' folder")

## 6. Quick Persistence Summary

In [ ]:
print("Metastasis Risk:", report['summary']['metastasis_risk'])
print("Number of Separatrix Crossings:", report['summary']['separatrix_crossings'])
print("\nPersistence Report (first 3 regimes):")
print(json.dumps(report['persistence']['persistence']['regimes'][:3], indent=2))

---

**Next Steps**

- Open `temporal_persistence_visualizer.html` to see the interactive timeline + Mapper view
- Open `mapper_visualizer.html` for the force-directed graph
- Try modifying the trace generation to simulate different metastasis behaviors